# 13 - Does training on two attacks help on a third?

Notebooks 10 and 11 trained the final recipe on one attack. Notebook 07b used all three. Here it is trained on **two: TextFooler and BERT-Attack**, then tested on all three attacks. PWWS is the held-out one, so the question is whether going from one attack to two improves the third.

| Trained on | PWWS (never seen) | already measured |
|---|---|---|
| TextFooler only | 72.3 | notebook 10 |
| BERT-Attack only | 72.8 | notebook 11 |
| TextFooler + BERT-Attack | ? | this notebook |

Caveat to state with the result: in the authors' protocol PWWS takes its replacement words from the same list as TextFooler and differs in its search, so PWWS is a weaker "third attack" than a truly different one. Notebook 14 adds a character-level attack (DeepWordBug) for that.

**Rules, as in notebooks 10 and 11.** No PWWS example is used anywhere, including indirect use: the attack examples come only from TextFooler and BERT-Attack runs against vanilla BERT, the encoder trained without attacks, the first TextFooler-only encoder, and this notebook's own BERT. The encoder's epoch is chosen on attacked validation sentences of the two training attacks. The clean-accuracy check rejects an encoder more than 1 point below BERT's own. One test run on the 1000 test sentences.

**Cost (L4, rough):** about 4.5 hours, including about 1 hour of attacks on training sentences. Cached on Drive; resumes after a disconnect. Run after 10 and 11.

In [ ]:
!pip -q install transformers datasets accelerate textattack tensorflow_hub
import os, sys, subprocess
REPO = 'https://github.com/shivanshgoyaliitk/for_shreyansh.git'
BRANCH = 'claude/magical-pasteur-ptp2k2'
DIR = '/content/for_shreyansh'
url = REPO
try:  # only needed if the repo is private: add a Colab secret named GITHUB_TOKEN
    from google.colab import userdata
    url = REPO.replace('https://', 'https://' + userdata.get('GITHUB_TOKEN') + '@')
except Exception:
    pass
if os.path.exists(DIR):
    subprocess.run(['git', '-C', DIR, 'pull', '-q'])
else:
    subprocess.run(['git', 'clone', '-q', '-b', BRANCH, url, DIR], check=True)
sys.path.insert(0, DIR)

In [ ]:
import os, json, torch, pandas as pd, tmd_lab as T
T.setup_env()                      # the authors' TextAttack fork for every attack
from google.colab import drive
drive.mount('/content/drive')

DATASET = 'ag_news'
BASE_K = 30
WORK = '/content/drive/MyDrive/tmd_run_paperft'   # same folder as notebooks 00-12
N_TEST = 1000
N_ROUND = 1000                     # fresh training sentences per attack, attacked against this notebook's own BERT
ALL = ('textfooler', 'bertattack', 'pwws')
ENC_EPOCHS = 30
W_DEN = dict(w_lat=1.0, w_rec=1.0, w_den=1.0, w_typ=0.1, w_adv=0.0, w_cls=0.0)
W_ADV = dict(w_lat=1.0, w_rec=1.0, w_den=1.0, w_typ=0.1, w_adv=1.0, w_cls=1.0)

lab = T.Lab(DATASET, work=WORK)
lab.finetune('base', **T.PAPER_FT)
E_tr, Y_tr = lab.embeddings('train', 'base')
E_va, Y_va = lab.embeddings('val', 'base')
E_te, Y_te = lab.embeddings('test', 'base')

START = json.load(open(lab._path('finetuned_advbert1/DONE')))['init']
RATE = (0.1, 0.3) if START == 'base' else tuple(json.load(open(lab._path('nb07_choice.json')))['rate'])
print('starting BERT, as in 07b:', START, '| noise rate:', RATE)
noise = T.NoiseAugmenter(lab.train_ds['text'], rate=RATE, seed=1)

def texts_from_logs(names):
    out, seen = [], set()
    for name in names:
        p = lab._path(name)
        assert os.path.exists(p), f'missing {p}'
        for r in map(json.loads, open(p)):
            if r['status'] == 'S' and (r['text'], r['adv_text']) not in seen:
                seen.add((r['text'], r['adv_text']))
                out.append((r['text'], r['adv_text'], r['label']))
    return out

TF_LOGS = ['advpairs_vanilla_textfooler_n1000_s1.jsonl',
           'advpairs_enc_den_textfooler_n1000_s2.jsonl',
           'advpairs_enc_adv_textfooler_n1000_s3.jsonl']
BA_LOGS = ['advpairs_vanilla_bertattack_n3000_s51.jsonl']
VAL_LOGS = ['advpairs_val_vanilla_textfooler_n500_s11.jsonl', 'advpairs_val_vanilla_bertattack_n500_s12.jsonl']
adv1 = texts_from_logs(TF_LOGS + BA_LOGS)
print(len(adv1), 'attacked training sentences (TextFooler + BERT-Attack)')

## Step 1: train BERT on clean, noisy and attacked training sentences (two attacks)

In [ ]:
lab.finetune_robust('mixbert1', init=START, epochs=2, lr=1e-5, noise_fn=noise, adv_texts=adv1)

## Step 2: attack this BERT with both attacks on fresh training sentences, then train once more

In [ ]:
lab.use_model('mixbert1')
own = T.Victim(lab, mode='none')
T.adversarial_pairs(lab, own, 'mixbert1', 'textfooler', N_ROUND, 61)
T.adversarial_pairs(lab, own, 'mixbert1', 'bertattack', N_ROUND, 62)
adv2 = texts_from_logs(TF_LOGS + BA_LOGS + [f'advpairs_mixbert1_textfooler_n{N_ROUND}_s61.jsonl',
                                            f'advpairs_mixbert1_bertattack_n{N_ROUND}_s62.jsonl'])
print(len(adv2), 'attacked training sentences in total')
lab.finetune_robust('mixbert2', init='mixbert1', epochs=1, lr=1e-5, noise_fn=noise, adv_texts=adv2)

## Step 3: the projection (Table 6 GAN + learned encoder, with the clean-accuracy check)

In [ ]:
lab.use_model('mixbert2')
Em_tr, _ = lab.embeddings('train', 'mixbert2')
Em_va, _ = lab.embeddings('val', 'mixbert2')
Em_te, _ = lab.embeddings('test', 'mixbert2')
Em_noise, _ = lab.embeddings('train', 'mixbert2', augment_fn=noise, name='train_noise')
gan_m = lab.get_gan('mixbert2', Em_tr, Em_va)
head_m = lab.model.classifier
adv_m = (T.embed_texts(lab, [r[1] for r in adv2]), T.embed_texts(lab, [r[0] for r in adv2]),
         torch.tensor([r[2] for r in adv2]))
val_rows = texts_from_logs(VAL_LOGS)
val_adv_m = (T.embed_texts(lab, [r[1] for r in val_rows]), torch.tensor([r[2] for r in val_rows]))

def get_enc(name, **kw):
    p = lab._path(f'projenc_{name}_L{gan_m.latent_dim}C{gan_m.code_dim}.pt')
    e = T.ProjEncoder(gan_m.latent_dim, gan_m.code_dim).to(T.DEV)
    if os.path.exists(p):
        e.load_state_dict(torch.load(p)); print('loaded', name)
        return e.eval()
    print('training', name)
    e = T.train_proj_encoder(gan_m, Em_tr, Y_tr, Em_va, Y_va, E_aug=Em_noise, head=head_m, epochs=ENC_EPOCHS, **kw)
    torch.save(e.state_dict(), p)
    return e

enc_den_m = get_enc('mix_enc_den', **W_DEN)

with torch.no_grad():
    bert_clean = (head_m(Em_va.to(T.DEV)).argmax(-1).cpu() == Y_va).float().mean().item()
def proj_clean(e):
    with torch.no_grad():
        th = T.project(gan_m, Em_va.to(T.DEV), enc=e, mode='encoder')[0]
        return (head_m(th).argmax(-1).cpu() == Y_va).float().mean().item()
for seed in (42, 1, 2, 3):
    name = 'mix_enc1' if seed == 42 else f'mix_enc1_s{seed}'
    enc1_m = get_enc(name, adv=adv_m, init=enc_den_m, select='best', val_adv=val_adv_m, seed=seed, **W_ADV)
    acc = proj_clean(enc1_m)
    print(f'{name}: projected clean val acc {acc:.4f} (BERT alone {bert_clean:.4f})')
    if acc >= bert_clean - 0.01:
        break
else:
    raise RuntimeError('no seed kept clean accuracy: send the log')
TAG = 'tf_ba_enc1' if seed == 42 else f'tf_ba_enc1_s{seed}'

## The one test run: two seen attacks (TextFooler, BERT-Attack) and one unseen (PWWS)

In [ ]:
lab.use_model('mixbert2')
v = T.Victim(lab, gan=gan_m, enc=enc1_m, mode='tmd', proj_mode='encoder')
T.evaluate(lab, v, TAG, Em_te, Y_te, attacks=ALL, n=N_TEST, need_log=True)
display(T.compare(lab, TAG, baseline_tag='baseline', n=N_TEST))
display(pd.DataFrame([T.mcnemar(lab, 'baseline', TAG, a, N_TEST) for a in ALL]))

def guarded(prefix):
    for t in (f'{prefix}_s1', f'{prefix}_s2', f'{prefix}_s3', prefix):
        p = T.results_path(lab, t, N_TEST)
        if os.path.exists(p) and json.load(open(p))['clean']['acc'] > 90:
            return t

P = dict(zip(T.METRICS, T.PAPER['ag_news']['TMD']))
rows = [{'defense': 'paper: TMD', 'attacks seen': 0, 'clean': P['clean'], 'TextFooler': P['textfooler'],
         'BERT-Attack': P['bertattack'], 'PWWS': P['pwws']}]
for tag, label, k in [('baseline', 'ours: TMD reproduced', 0), (guarded('tf_only_enc1'), 'trained on TextFooler only', 1),
                      (guarded('ba_only_enc1'), 'trained on BERT-Attack only', 1),
                      (TAG, 'trained on TextFooler + BERT-Attack', 2), ('advbert2_enc1', 'trained on all three', 3)]:
    if tag and os.path.exists(T.results_path(lab, tag, N_TEST)):
        r = json.load(open(T.results_path(lab, tag, N_TEST)))
        rows.append({'defense': label, 'attacks seen': k, 'clean': round(r['clean']['acc'], 2),
                     **{T.METRIC_LABEL[a]: round(r[a]['acc'], 1) for a in ALL if a in r}})
display(pd.DataFrame(rows))
print('PWWS was never seen by rows with 1 or 2 attacks. Compare its value down the "attacks seen" column.')